# Experimental XRD to MP-ID Predictions

This notebook loads the experimental XRD pattern in `exp_data/intensity.csv`, runs the repository's trained XDecomposer model, and reports the most likely Materials Project IDs (MP-IDs) for the predicted phases.

Run the cells from the repository root. The model checkpoints and `MP500.db` included with this project are used locally; no network request is required.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Locate the repository even when the notebook is opened from another directory.
root = Path.cwd().resolve()
if not (root / 'exp_data' / 'intensity.csv').is_file():
    root = next(
        parent for parent in [root, *root.parents]
        if (parent / 'exp_data' / 'intensity.csv').is_file()
    )

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

data_path = root / 'exp_data' / 'intensity.csv'
print(f'Repository: {root}')
print(f'Experimental data: {data_path}')

In [ ]:
# The CSV has no header: column 1 is 2theta (degrees), column 2 is intensity.
xrd = pd.read_csv(data_path, header=None, names=['two_theta', 'intensity'])
xrd = xrd.apply(pd.to_numeric, errors='raise').dropna().sort_values('two_theta')

if len(xrd) < 2:
    raise ValueError('The experimental XRD CSV must contain at least two rows.')
if not xrd['two_theta'].is_monotonic_increasing or xrd['two_theta'].duplicated().any():
    raise ValueError('2theta values must be strictly increasing.')

display(xrd.head())
print(f'Loaded {len(xrd):,} XRD points from {xrd.two_theta.min():.2f} to {xrd.two_theta.max():.2f} degrees.')

plt.figure(figsize=(12, 3.5))
plt.plot(xrd['two_theta'], xrd['intensity'], linewidth=0.8)
plt.xlabel(r'2$\theta$ (degrees)')
plt.ylabel('Intensity')
plt.title('Experimental XRD pattern')
plt.tight_layout()
plt.show()

## Run local inference

The first run builds (and caches) the reference XRD bank from `MP500.db`, so it can take a little longer than subsequent runs. `top_k` controls how many candidate MP-IDs are retained for each predicted phase.

In [ ]:
from api.main import DecompositionService, DecomposeRequest

top_k = 5
service = DecompositionService()
service.load()

request = DecomposeRequest(
    two_theta=xrd['two_theta'].astype(float).tolist(),
    intensities=xrd['intensity'].astype(float).tolist(),
)
prediction = service.decompose(request, top_k=top_k, include_patterns=False)
print(f'Model input grid: {prediction.xrd_length:,} points')

In [ ]:
# The first row for each phase is its best MP-ID prediction.
phase_results = pd.DataFrame([
    {
        'phase': phase.phase_index + 1,
        'active': phase.active,
        'activity_probability': phase.activity_probability,
        'predicted_mp_id': phase.mp_id,
        'match_score': phase.match_score,
    }
    for phase in prediction.phases
]).sort_values(['active', 'activity_probability'], ascending=[False, False])

display(phase_results.style.format({
    'activity_probability': '{:.3f}',
    'match_score': '{:.3f}',
}))

active_mp_ids = phase_results.loc[phase_results['active'], 'predicted_mp_id'].tolist()
print('Predicted active MP-ID(s):', ', '.join(active_mp_ids) if active_mp_ids else 'None')

In [ ]:
# Inspect alternative database matches for every predicted phase.
candidate_results = pd.concat([
    pd.DataFrame(phase.alternatives).assign(
        phase=phase.phase_index + 1,
        activity_probability=phase.activity_probability,
    )
    for phase in prediction.phases
], ignore_index=True)
candidate_results = candidate_results[['phase', 'activity_probability', 'mp_id', 'match_score']]
display(candidate_results.style.format({
    'activity_probability': '{:.3f}',
    'match_score': '{:.3f}',
}))

output_path = root / 'exp_data' / 'mpid_predictions.csv'
phase_results.to_csv(output_path, index=False)
print(f'Saved best MP-ID predictions to: {output_path}')